### Original notebook cell 2


In [ ]:

from pathlib import Path
import zipfile
import pandas as pd

# Find the research project directory
project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

zip_path = project_root / "data" / "raw" / "prosper-loans.zip"

print("Dataset exists:", zip_path.exists())
print("Dataset path:", zip_path)

# Inspect the ZIP archive without extracting it
with zipfile.ZipFile(zip_path, "r") as archive:
    print("\nFiles inside the archive:")
    for file in archive.namelist():
        print(file)


### Original notebook cell 3


In [ ]:

# Define provisional column names
columns = [
    "source_id",
    "target_id",
    "edge_weight",
    "timestamp",
    "amount",
    "loan_status",
    "interest_rate_1",
    "interest_rate_2",
    "credit_grade"
]

# Read only the first 10 records
with zipfile.ZipFile(zip_path, "r") as archive:
    with archive.open("out.prosper-loans") as file:
        prosper_sample = pd.read_csv(
            file,
            sep=r"\s+",
            comment="%",
            header=None,
            names=columns,
            nrows=10
        )

print("Dataset sample shape:", prosper_sample.shape)
display(prosper_sample)


### Original notebook cell 5


In [ ]:

# Load the complete Prosper lending network
with zipfile.ZipFile(zip_path, "r") as archive:
    with archive.open("out.prosper-loans") as file:
        prosper_df = pd.read_csv(
            file,
            sep=r"\s+",
            comment="%",
            header=None,
            names=columns
        )

print("Dataset shape:", prosper_df.shape)

print("\nColumn data types:")
print(prosper_df.dtypes)

print("\nMissing values:")
print(prosper_df.isnull().sum())

print("\nFirst five records:")
display(prosper_df.head())


### Original notebook cell 13


In [ ]:

# Convert Unix timestamps into readable UTC dates
prosper_df["loan_datetime"] = pd.to_datetime(
    prosper_df["timestamp"],
    unit="s",
    utc=True
)

print("Earliest recorded date:")
print(prosper_df["loan_datetime"].min())

print("\nLatest recorded date:")
print(prosper_df["loan_datetime"].max())

# Examine whether timestamps contain time-of-day information
unique_times = prosper_df[
    "loan_datetime"
].dt.strftime("%H:%M:%S").value_counts()

print("\nMost common recorded times:")
display(unique_times.head(10))

print("\nNumber of distinct times of day:", len(unique_times))

# Count distinct timestamps
print(
    "\nNumber of distinct timestamps:",
    prosper_df["timestamp"].nunique()
)

# Display a sample of readable dates
display(
    prosper_df[
        ["target_id", "timestamp", "loan_datetime"]
    ].head(10)
)


### Original notebook cell 17


In [ ]:

# Create a temporary table of potential borrowing events
event_df = (
    prosper_df.groupby(
        ["target_id", "timestamp"],
        as_index=False
    )
    .agg(
        lending_records=("source_id", "size"),
        unique_lenders=("source_id", "nunique"),
        total_recorded_amount=("amount", "sum"),
        loan_status=("loan_status", "first"),
        interest_rate_1=("interest_rate_1", "first"),
        interest_rate_2=("interest_rate_2", "first"),
        credit_grade=("credit_grade", "first")
    )
)

print("Potential borrowing events:", len(event_df))

print("\nFinancial attribute summary:")
display(
    event_df[
        [
            "lending_records",
            "unique_lenders",
            "total_recorded_amount",
            "interest_rate_1",
            "interest_rate_2"
        ]
    ].describe()
)

print("\nCredit grade distribution:")
display(
    event_df["credit_grade"].value_counts()
)

print("\nEvent-level loan status distribution:")
display(
    event_df["loan_status"].value_counts()
)

print("\nFirst five potential borrowing events:")
display(event_df.head())


### Original notebook cell 19


In [ ]:

# Select potential borrowing events with resolved outcomes
resolved_statuses = ["Paid", "Defaulted", "Charge-off"]

resolved_events = event_df[
    event_df["loan_status"].isin(resolved_statuses)
].copy()

# Create a provisional binary target for exploration
resolved_events["default_flag"] = resolved_events[
    "loan_status"
].isin(["Defaulted", "Charge-off"]).astype(int)

print("Resolved potential borrowing events:", len(resolved_events))

print("\nBinary outcome distribution:")
display(
    resolved_events["default_flag"]
    .value_counts()
    .rename(index={0: "Non-default", 1: "Default"})
)

# Examine default proportions by credit grade
grade_summary = (
    resolved_events.groupby("credit_grade")
    .agg(
        events=("default_flag", "size"),
        defaults=("default_flag", "sum"),
        default_rate=("default_flag", "mean")
    )
    .sort_values("default_rate")
)

grade_summary["default_rate"] = (
    grade_summary["default_rate"] * 100
).round(2)

print("\nDefault proportion by credit grade (%):")
display(grade_summary)


### Original notebook cell 21


In [ ]:

# Sort potential borrowing events chronologically
event_history = event_df.sort_values(
    ["target_id", "timestamp"]
).copy()

# Count earlier recorded events for each borrower
event_history["previous_event_count"] = (
    event_history.groupby("target_id").cumcount()
)

# Identify whether a borrower has prior recorded activity
event_history["has_previous_event"] = (
    event_history["previous_event_count"] > 0
)

print("Total potential borrowing events:", len(event_history))

print(
    "\nEvents with previous borrower activity:",
    event_history["has_previous_event"].sum()
)

print(
    "Events without previous borrower activity:",
    (~event_history["has_previous_event"]).sum()
)

print("\nPrevious event count distribution:")
display(
    event_history["previous_event_count"].describe()
)

# Examine historical activity among resolved outcomes
resolved_history = event_history[
    event_history["loan_status"].isin(
        ["Paid", "Defaulted", "Charge-off"]
    )
].copy()

print("\nResolved events with previous borrower activity:")
display(
    resolved_history["has_previous_event"].value_counts()
)

print("\nExamples of repeat borrowing histories:")
display(
    event_history[
        event_history["previous_event_count"] > 0
    ][
        [
            "target_id",
            "timestamp",
            "previous_event_count",
            "loan_status",
            "credit_grade"
        ]
    ].head(10)
)


### Original notebook cell 23


In [ ]:

# Count how many distinct lenders have funded
# borrowers on dates earlier than each event

# One record per lender, borrower and date
unique_funding = prosper_df[
    ["source_id", "target_id", "timestamp"]
].drop_duplicates()

# First recorded lending date for each lender
first_lender_date = (
    unique_funding.groupby("source_id")["timestamp"]
    .min()
    .rename("first_lender_date")
)

# Attach each lender's first recorded date
funding_history = unique_funding.join(
    first_lender_date,
    on="source_id"
)

# A lender has earlier recorded activity only if
# their first date precedes the current event date
funding_history["historically_active_lender"] = (
    funding_history["first_lender_date"]
    < funding_history["timestamp"]
)

# Summarize historical lender coverage per event
historical_coverage = (
    funding_history.groupby(["target_id", "timestamp"])
    .agg(
        total_unique_lenders=("source_id", "size"),
        lenders_with_prior_activity=(
            "historically_active_lender", "sum"
        )
    )
    .reset_index()
)

historical_coverage["has_historical_lender"] = (
    historical_coverage["lenders_with_prior_activity"] > 0
)

# Restrict summary to resolved events
resolved_coverage = resolved_events[
    ["target_id", "timestamp"]
].merge(
    historical_coverage,
    on=["target_id", "timestamp"],
    how="left",
    validate="one_to_one"
)

print("Resolved events:", len(resolved_coverage))

print("\nEvents with at least one historically active lender:")
display(
    resolved_coverage[
        "has_historical_lender"
    ].value_counts()
)

print("\nHistorical lender counts per resolved event:")
display(
    resolved_coverage[
        "lenders_with_prior_activity"
    ].describe()
)

print("\nPercentage of resolved events with historical lenders:")
print(
    round(
        resolved_coverage[
            "has_historical_lender"
        ].mean() * 100, 2
    ),
    "%"
)


### Original notebook cell 25


In [ ]:

# Convert event timestamps to calendar years
event_df["year"] = pd.to_datetime(
    event_df["timestamp"],
    unit="s",
    utc=True
).dt.year

# Count all potential borrowing events by year
events_by_year = (
    event_df.groupby("year")
    .size()
    .rename("all_events")
)

# Count resolved borrowing events by year
resolved_by_year = (
    event_df[
        event_df["loan_status"].isin(
            ["Paid", "Defaulted", "Charge-off"]
        )
    ]
    .groupby("year")
    .size()
    .rename("resolved_events")
)

# Count unique lenders active each year
prosper_df["year"] = prosper_df[
    "loan_datetime"
].dt.year

lenders_by_year = (
    prosper_df.groupby("year")["source_id"]
    .nunique()
    .rename("unique_lenders")
)

# Combine annual statistics
annual_summary = pd.concat(
    [
        events_by_year,
        resolved_by_year,
        lenders_by_year
    ],
    axis=1
).fillna(0).astype(int)

annual_summary["resolved_percentage"] = (
    annual_summary["resolved_events"]
    / annual_summary["all_events"] * 100
).round(2)

print("Annual lending activity:")
display(annual_summary)


### Original notebook cell 27


In [ ]:

# Create a monthly period for each potential borrowing event
event_df["month"] = pd.to_datetime(
    event_df["timestamp"],
    unit="s",
    utc=True
).dt.to_period("M")

# Identify whether the recorded outcome is resolved
event_df["is_resolved"] = event_df[
    "loan_status"
].isin(["Paid", "Defaulted", "Charge-off"])

# Summarize monthly outcome coverage
monthly_summary = (
    event_df.groupby("month")
    .agg(
        all_events=("target_id", "size"),
        resolved_events=("is_resolved", "sum")
    )
)

monthly_summary["resolved_percentage"] = (
    monthly_summary["resolved_events"]
    / monthly_summary["all_events"] * 100
).round(2)

print("Monthly outcome coverage during 2008:")
display(
    monthly_summary.loc["2008-01":"2008-12"]
)

print("\nMonthly outcome coverage during 2009:")
display(
    monthly_summary.loc["2009-01":"2009-12"]
)


### Original notebook cell 38


In [ ]:

# Step 23: Create unique identifiers for provisional borrowing events

# Start from the existing event-level dataset
event_data = event_df.copy()

# Sort events chronologically, then by borrower ID
event_data = event_data.sort_values(
    ["timestamp", "target_id"]
).reset_index(drop=True)

# Assign one unique identifier to each borrower-date group
event_data["event_id"] = range(len(event_data))

# Verify identifier uniqueness
print("Total borrowing events:", len(event_data))

print(
    "Unique event IDs:",
    event_data["event_id"].nunique()
)

print(
    "Duplicate borrower-date combinations:",
    event_data.duplicated(
        subset=["target_id", "timestamp"]
    ).sum()
)

print(
    "Duplicate event IDs:",
    event_data["event_id"].duplicated().sum()
)

# Inspect the resulting table
display(event_data.head(10))


### Original notebook cell 40


In [ ]:

# Step 24: Link original lending records to event IDs

event_lookup = event_data[
    ["target_id", "timestamp", "event_id"]
].copy()

# Attach the provisional event ID to every original record
lending_edges = prosper_df.merge(
    event_lookup,
    on=["target_id", "timestamp"],
    how="left",
    validate="many_to_one"
)

# Validate that all original records are preserved
print("Original lending records:", len(prosper_df))
print("Linked lending records:", len(lending_edges))

print(
    "Records without event IDs:",
    lending_edges["event_id"].isna().sum()
)

print(
    "Unique linked events:",
    lending_edges["event_id"].nunique()
)

print(
    "All records preserved:",
    len(lending_edges) == len(prosper_df)
)

# Display a sample of the linked records
display(
    lending_edges[
        [
            "source_id",
            "target_id",
            "timestamp",
            "amount",
            "event_id"
        ]
    ].head(10)
)


### Original notebook cell 42


In [ ]:

# Step 25: Define the provisional modelling population

# Select the candidate modelling period
start_date = pd.Timestamp("2006-01-01", tz="UTC")
end_date = pd.Timestamp("2008-09-01", tz="UTC")

event_dates = pd.to_datetime(
    event_data["timestamp"],
    unit="s",
    utc=True
)

period_mask = (
    (event_dates >= start_date)
    & (event_dates < end_date)
)

candidate_events = event_data.loc[period_mask].copy()

# Keep only the provisional resolved outcome categories
resolved_statuses = ["Paid", "Defaulted", "Charge-off"]

model_events = candidate_events[
    candidate_events["loan_status"].isin(resolved_statuses)
].copy()

# Create the binary target
model_events["default_flag"] = (
    model_events["loan_status"]
    .isin(["Defaulted", "Charge-off"])
    .astype(int)
)

print("Events in candidate period:", len(candidate_events))
print("Resolved modelling events:", len(model_events))

print(
    "Excluded unresolved/other events:",
    len(candidate_events) - len(model_events)
)

print("\nTarget distribution:")
display(
    model_events["default_flag"]
    .value_counts()
    .rename(index={0: "Non-default", 1: "Default"})
)

print("\nTarget percentages:")
display(
    (
        model_events["default_flag"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
        .rename(index={0: "Non-default", 1: "Default"})
    )
)

print("\nSelected period:")
print("Earliest:", event_dates.loc[model_events.index].min())
print("Latest:", event_dates.loc[model_events.index].max())

print("\nUnique modelling event IDs:")
print(model_events["event_id"].nunique())


### Original notebook cell 44


In [ ]:

# Step 26: Examine provisional chronological partitions

split_data = model_events.copy()

split_data["event_date"] = pd.to_datetime(
    split_data["timestamp"],
    unit="s",
    utc=True
)

train_end = pd.Timestamp("2008-01-01", tz="UTC")
validation_end = pd.Timestamp("2008-05-01", tz="UTC")

split_data["partition"] = "test"

split_data.loc[
    split_data["event_date"] < train_end,
    "partition"
] = "train"

split_data.loc[
    (split_data["event_date"] >= train_end)
    & (split_data["event_date"] < validation_end),
    "partition"
] = "validation"

# Count events and outcomes in each partition
split_summary = (
    split_data.groupby("partition")
    .agg(
        events=("event_id", "size"),
        defaults=("default_flag", "sum"),
        default_rate=("default_flag", "mean"),
        first_date=("event_date", "min"),
        last_date=("event_date", "max")
    )
)

split_summary["non_defaults"] = (
    split_summary["events"] - split_summary["defaults"]
)

split_summary["default_rate"] = (
    split_summary["default_rate"] * 100
).round(2)

split_summary = split_summary[
    [
        "events",
        "non_defaults",
        "defaults",
        "default_rate",
        "first_date",
        "last_date"
    ]
]

display(
    split_summary.reindex(
        ["train", "validation", "test"]
    )
)

print("\nTotal partitioned events:", len(split_data))

print(
    "Unique partitioned event IDs:",
    split_data["event_id"].nunique()
)

print(
    "Missing partition assignments:",
    split_data["partition"].isna().sum()
)


### Original notebook cell 48


In [ ]:

# Step 28: Inspect candidate modelling features

print("Modelling dataset shape:")
print(split_data.shape)

print("\nAvailable columns:")
print(split_data.columns.tolist())

print("\nMissing values by column:")
display(
    split_data.isna().sum().sort_values(ascending=False)
)

print("\nCandidate numerical feature summary:")
candidate_numeric = [
    "total_recorded_amount",
    "unique_lenders",
    "lending_records",
    "interest_rate_1",
    "interest_rate_2"
]

display(
    split_data[candidate_numeric].describe()
)

print("\nCredit grade distribution:")
display(
    split_data["credit_grade"].value_counts()
)

print("\nCredit grades by partition:")
display(
    pd.crosstab(
        split_data["partition"],
        split_data["credit_grade"]
    ).reindex(
        ["train", "validation", "test"]
    )
)


### Original notebook cell 50


In [ ]:

# Step 29: Prepare the initial shared feature set

numeric_features = [
    "total_recorded_amount",
    "unique_lenders",
    "lending_records"
]

categorical_features = [
    "credit_grade"
]

feature_columns = numeric_features + categorical_features

# Separate the observations by chronological partition
train_data = split_data[
    split_data["partition"] == "train"
].copy()

validation_data = split_data[
    split_data["partition"] == "validation"
].copy()

test_data = split_data[
    split_data["partition"] == "test"
].copy()

# Prepare predictors and target labels
X_train = train_data[feature_columns].copy()
X_validation = validation_data[feature_columns].copy()
X_test = test_data[feature_columns].copy()

y_train = train_data["default_flag"].copy()
y_validation = validation_data["default_flag"].copy()
y_test = test_data["default_flag"].copy()

print("Selected features:", feature_columns)

print("\nFeature matrix shapes:")
print("Training:", X_train.shape)
print("Validation:", X_validation.shape)
print("Testing:", X_test.shape)

print("\nTarget vector shapes:")
print("Training:", y_train.shape)
print("Validation:", y_validation.shape)
print("Testing:", y_test.shape)

print("\nTraining feature sample:")
display(X_train.head())

print("\nTarget distributions:")
print("Training:", y_train.value_counts().to_dict())
print("Validation:", y_validation.value_counts().to_dict())
print("Testing:", y_test.value_counts().to_dict())


### Original notebook cell 52


In [ ]:

# Step 30: Build the Logistic Regression pipeline

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Numerical preprocessing
numeric_transformer = StandardScaler()

# Categorical preprocessing
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

# Build the complete model pipeline
lr_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

print("Logistic Regression pipeline created successfully.")

display(lr_pipeline)


### Original notebook cell 54


In [ ]:

# Step 31: Train and validate Logistic Regression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# Fit preprocessing and classifier on training data only
lr_pipeline.fit(X_train, y_train)

# Generate validation predictions
y_val_pred = lr_pipeline.predict(X_validation)

# Probability of the default class (1)
y_val_proba = lr_pipeline.predict_proba(X_validation)[:, 1]

# Calculate validation metrics
validation_metrics = {
    "Accuracy": accuracy_score(y_validation, y_val_pred),
    "Precision (Default)": precision_score(
        y_validation, y_val_pred, zero_division=0
    ),
    "Recall (Default)": recall_score(
        y_validation, y_val_pred, zero_division=0
    ),
    "F1-score (Default)": f1_score(
        y_validation, y_val_pred, zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_validation, y_val_proba
    ),
    "PR-AUC (Average Precision)": average_precision_score(
        y_validation, y_val_proba
    )
}

print("LOGISTIC REGRESSION - VALIDATION RESULTS")
print("-" * 45)

for metric, value in validation_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_validation, y_val_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_validation,
        y_val_pred,
        target_names=["Non-default", "Default"],
        zero_division=0
    )
)


### Original notebook cell 60


In [ ]:

# Step 34: Prepare unique lender-event graph connections

# Preserve the original funding records
graph_edges = (
    lending_edges[
        ["source_id", "event_id", "timestamp"]
    ]
    .drop_duplicates(
        subset=["source_id", "event_id"]
    )
    .copy()
)

# Rename columns for clarity
graph_edges = graph_edges.rename(
    columns={
        "source_id": "lender_id"
    }
)

print("Original lending records:", len(lending_edges))
print("Unique lender-event edges:", len(graph_edges))

print(
    "Repeated records represented by existing edges:",
    len(lending_edges) - len(graph_edges)
)

print(
    "\nDuplicate lender-event pairs:",
    graph_edges.duplicated(
        subset=["lender_id", "event_id"]
    ).sum()
)

print(
    "\nUnique lenders:",
    graph_edges["lender_id"].nunique()
)

print(
    "Unique connected events:",
    graph_edges["event_id"].nunique()
)

print("\nSample graph edges:")
display(graph_edges.head(10))


### Original notebook cell 62


In [ ]:

# Step 35: Create consecutive graph node indices

import pandas as pd

# Create lender node mapping
unique_lenders = sorted(
    graph_edges["lender_id"].unique()
)

lender_mapping = pd.DataFrame({
    "lender_id": unique_lenders,
    "lender_node_index": range(len(unique_lenders))
})

# Create event node mapping
event_mapping = event_data[
    ["event_id", "target_id", "timestamp"]
].copy()

event_mapping["event_node_index"] = (
    event_mapping["event_id"]
)

# Attach lender node indices to graph edges
indexed_graph_edges = graph_edges.merge(
    lender_mapping,
    on="lender_id",
    how="left",
    validate="many_to_one"
)

# Attach event node indices
indexed_graph_edges = indexed_graph_edges.merge(
    event_mapping[
        ["event_id", "event_node_index"]
    ],
    on="event_id",
    how="left",
    validate="many_to_one"
)

print("Total lender nodes:", len(lender_mapping))
print("Total event nodes:", len(event_mapping))

print("\nIndexed graph edges:", len(indexed_graph_edges))

print(
    "Missing lender indices:",
    indexed_graph_edges["lender_node_index"].isna().sum()
)

print(
    "Missing event indices:",
    indexed_graph_edges["event_node_index"].isna().sum()
)

print(
    "Unique lender indices:",
    lender_mapping["lender_node_index"].nunique()
)

print(
    "Unique event indices:",
    event_mapping["event_node_index"].nunique()
)

print("\nSample indexed graph edges:")
display(
    indexed_graph_edges[
        [
            "lender_id",
            "lender_node_index",
            "event_id",
            "event_node_index",
            "timestamp"
        ]
    ].head(10)
)


### Original notebook cell 68


In [ ]:

# Step 38: Prepare event-node features using training-fitted preprocessing

import numpy as np
import pandas as pd

# Use the existing Logistic Regression preprocessing pipeline
# It was already fitted on X_train only.

event_feature_columns = (
    numeric_features + categorical_features
)

# Keep modelling events in event_id order
graph_model_events = (
    split_data
    .sort_values("event_id")
    .reset_index(drop=True)
    .copy()
)

# Transform event features using the fitted training preprocessor
event_features = lr_pipeline.named_steps[
    "preprocessor"
].transform(
    graph_model_events[event_feature_columns]
)

# Convert sparse output to a dense array if necessary
if hasattr(event_features, "toarray"):
    event_features = event_features.toarray()

event_features = np.asarray(
    event_features,
    dtype=np.float32
)

# Create an event_id-to-feature-row mapping
event_feature_mapping = pd.DataFrame({
    "event_id": graph_model_events["event_id"],
    "feature_row_index": np.arange(
        len(graph_model_events)
    )
})

print("EVENT FEATURE PREPARATION")
print("-------------------------")

print("Number of labelled events:", len(graph_model_events))

print("Event feature matrix shape:", event_features.shape)

print("Missing feature values:", np.isnan(event_features).sum())

print("Infinite feature values:", np.isinf(event_features).sum())

print(
    "Duplicate event IDs:",
    event_feature_mapping["event_id"].duplicated().sum()
)

print("\nFirst five feature vectors:")
display(
    pd.DataFrame(event_features).head()
)

print("\nEvent feature mapping:")
display(event_feature_mapping.head())


### Original notebook cell 70


In [ ]:

# Step 39: Prepare initial lender-node features

import numpy as np

# Number of lender nodes
num_lenders = len(lender_mapping)

# Assign one constant feature to every lender
lender_features = np.ones(
    (num_lenders, 1),
    dtype=np.float32
)

# Verify that graph node indices are consecutive
expected_indices = np.arange(num_lenders)

actual_indices = (
    lender_mapping["lender_node_index"]
    .to_numpy()
)

indices_valid = np.array_equal(
    actual_indices,
    expected_indices
)

print("LENDER FEATURE PREPARATION")
print("--------------------------")

print("Number of lenders:", num_lenders)

print(
    "Lender feature matrix shape:",
    lender_features.shape
)

print(
    "Consecutive lender indices:",
    indices_valid
)

print(
    "Missing lender feature values:",
    np.isnan(lender_features).sum()
)

print(
    "Infinite lender feature values:",
    np.isinf(lender_features).sum()
)

print("\nFirst five lender feature vectors:")
print(lender_features[:5])

print("\nFirst five lender mappings:")
display(lender_mapping.head())


### Original notebook cell 84


In [ ]:

# Step 43: Prepare features for all borrowing-event nodes

import numpy as np
import pandas as pd

# All borrowing events, ordered by their existing graph node indices
all_graph_events = (
    event_data
    .sort_values("event_id")
    .reset_index(drop=True)
    .copy()
)

# Use the preprocessor fitted on the training partition only
graph_preprocessor = lr_pipeline.named_steps["preprocessor"]

all_event_features = graph_preprocessor.transform(
    all_graph_events[feature_columns]
)

# Convert to a dense NumPy array if necessary
if hasattr(all_event_features, "toarray"):
    all_event_features = all_event_features.toarray()

all_event_features = np.asarray(
    all_event_features,
    dtype=np.float32
)

# Verify that event IDs correspond to graph node indices
event_indices_valid = np.array_equal(
    all_graph_events["event_id"].to_numpy(),
    np.arange(len(all_graph_events))
)

print("COMPLETE GRAPH EVENT FEATURES")
print("-----------------------------")
print("Total event nodes:", len(all_graph_events))
print("Feature matrix shape:", all_event_features.shape)
print("Feature columns:", feature_columns)
print("Event indices aligned:", event_indices_valid)
print("Missing feature values:", np.isnan(all_event_features).sum())
print("Infinite feature values:", np.isinf(all_event_features).sum())

assert event_indices_valid, "Event node indices are not aligned!"
assert np.isfinite(all_event_features).all(), "Invalid feature values found!"

print("\nComplete event feature preparation successful!")


### Original notebook cell 86


In [ ]:

# Step 44: Convert graph node features into PyTorch tensors

import torch

# Borrowing-event node features
event_x = torch.from_numpy(
    all_event_features
).float()

# Lender node features
lender_x = torch.from_numpy(
    lender_features
).float()

print("GRAPH NODE FEATURE TENSORS")
print("--------------------------")

print("Borrowing-event tensor shape:", tuple(event_x.shape))
print("Lender tensor shape:", tuple(lender_x.shape))

print("\nEvent tensor dtype:", event_x.dtype)
print("Lender tensor dtype:", lender_x.dtype)

print("\nEvent tensor device:", event_x.device)
print("Lender tensor device:", lender_x.device)

print("\nEvent tensor finite:", torch.isfinite(event_x).all().item())
print("Lender tensor finite:", torch.isfinite(lender_x).all().item())

assert event_x.shape == (43572, 11)
assert lender_x.shape == (59046, 1)
assert torch.isfinite(event_x).all()
assert torch.isfinite(lender_x).all()

print("\nGraph node tensor preparation successful!")


### Original notebook cell 92


In [ ]:

# Step 47: Prepare supervised labels and partition masks

import numpy as np
import torch

num_events = len(all_graph_events)

# -1 means the event has no eligible supervised label
event_y = torch.full(
    (num_events,),
    -1,
    dtype=torch.long
)

train_node_mask = torch.zeros(num_events, dtype=torch.bool)
validation_node_mask = torch.zeros(num_events, dtype=torch.bool)
test_node_mask = torch.zeros(num_events, dtype=torch.bool)

# Assign labels and masks using the established event IDs
event_ids = torch.tensor(
    split_data["event_id"].to_numpy().copy(),
    dtype=torch.long
)

labels = torch.tensor(
    split_data["default_flag"].to_numpy().copy(),
    dtype=torch.long
)

event_y[event_ids] = labels

train_ids = torch.tensor(
    split_data.loc[
        split_data["partition"] == "train", "event_id"
    ].to_numpy().copy(),
    dtype=torch.long
)

validation_ids = torch.tensor(
    split_data.loc[
        split_data["partition"] == "validation", "event_id"
    ].to_numpy().copy(),
    dtype=torch.long
)

test_ids = torch.tensor(
    split_data.loc[
        split_data["partition"] == "test", "event_id"
    ].to_numpy().copy(),
    dtype=torch.long
)

train_node_mask[train_ids] = True
validation_node_mask[validation_ids] = True
test_node_mask[test_ids] = True

# Validate label and partition consistency
supervised_mask = (
    train_node_mask |
    validation_node_mask |
    test_node_mask
)

assert supervised_mask.sum().item() == len(split_data)
assert torch.all(event_y[supervised_mask] >= 0)
assert torch.all(event_y[~supervised_mask] == -1)

assert not torch.any(train_node_mask & validation_node_mask)
assert not torch.any(train_node_mask & test_node_mask)
assert not torch.any(validation_node_mask & test_node_mask)

print("SUPERVISED GRAPH LABELS AND MASKS")
print("---------------------------------")
print("Total graph events:", num_events)
print("Supervised events:", supervised_mask.sum().item())
print("Unlabelled/excluded events:", (~supervised_mask).sum().item())

print("\nPartition counts:")
print("Training:", train_node_mask.sum().item())
print("Validation:", validation_node_mask.sum().item())
print("Testing:", test_node_mask.sum().item())

print("\nDefault counts:")
print("Training:", event_y[train_node_mask].sum().item())
print("Validation:", event_y[validation_node_mask].sum().item())
print("Testing:", event_y[test_node_mask].sum().item())

print("\nSupervised label preparation successful!")


### Original notebook cell 98


In [ ]:

# Step 50: Prepare time-ordered lender-event adjacency

import numpy as np
import pandas as pd

# Extract the necessary graph relationship information
temporal_adjacency = indexed_graph_edges[
    ["lender_node_index", "event_node_index", "timestamp"]
].copy()

# Sort by lender, then chronological event timestamp
temporal_adjacency = (
    temporal_adjacency
    .sort_values(
        ["lender_node_index", "timestamp", "event_node_index"]
    )
    .reset_index(drop=True)
)

# Identify where each lender's adjacency list begins and ends
lender_ids = temporal_adjacency[
    "lender_node_index"
].to_numpy(dtype=np.int64)

lender_event_ids = temporal_adjacency[
    "event_node_index"
].to_numpy(dtype=np.int64)

lender_event_times = temporal_adjacency[
    "timestamp"
].to_numpy(dtype=np.int64)

# CSR-style offsets for fast lender-neighbour lookup
lender_edge_counts = np.bincount(
    lender_ids,
    minlength=len(lender_mapping)
)

lender_offsets = np.zeros(
    len(lender_mapping) + 1,
    dtype=np.int64
)

lender_offsets[1:] = np.cumsum(lender_edge_counts)

# Validate adjacency structure
assert lender_offsets[-1] == len(temporal_adjacency)
assert len(lender_event_ids) == len(indexed_graph_edges)
assert np.all(np.diff(lender_offsets) >= 0)

# Check chronological ordering within lender groups
same_lender = lender_ids[1:] == lender_ids[:-1]

assert np.all(
    lender_event_times[1:][same_lender]
    >= lender_event_times[:-1][same_lender]
)

print("TEMPORAL LENDER ADJACENCY")
print("-------------------------")
print("Lenders:", len(lender_mapping))
print("Unique funding relationships:", len(temporal_adjacency))
print("Lender offsets:", len(lender_offsets))
print("Event indices:", len(lender_event_ids))
print("Timestamps:", len(lender_event_times))

print("\nFirst five lender edge counts:")
print(lender_edge_counts[:5])

print("\nChronological ordering verified:", True)
print("Adjacency preparation successful!")


### Original notebook cell 100


In [ ]:

# Step 51: Test time-restricted historical-neighbour retrieval

import numpy as np

def get_prior_lender_events(lender_index, target_timestamp):
    """
    Return event indices connected to a lender strictly
    before the target timestamp.
    """
    start = lender_offsets[lender_index]
    end = lender_offsets[lender_index + 1]

    times = lender_event_times[start:end]
    events = lender_event_ids[start:end]

    # Find the first position at or after the target timestamp
    cutoff = np.searchsorted(
        times,
        target_timestamp,
        side="left"
    )

    return events[:cutoff]


# Select the earliest supervised validation event
sample_event = (
    split_data.loc[split_data["partition"] == "validation"]
    .sort_values(["timestamp", "event_id"])
    .iloc[0]
)

sample_event_id = int(sample_event["event_id"])
sample_timestamp = int(sample_event["timestamp"])

# Identify lenders funding the selected event
sample_lenders = temporal_adjacency.loc[
    temporal_adjacency["event_node_index"] == sample_event_id,
    "lender_node_index"
].to_numpy(dtype=np.int64)

# Test the first five participating lenders
selected_lenders = sample_lenders[:5]

print("HISTORICAL NEIGHBOUR RETRIEVAL")
print("------------------------------")
print("Sample event ID:", sample_event_id)
print(
    "Sample event date:",
    pd.to_datetime(sample_timestamp, unit="s", utc=True)
)
print("Total lenders funding event:", len(sample_lenders))

print("\nHistorical events for first five lenders:")

for lender in selected_lenders:
    prior_events = get_prior_lender_events(
        int(lender),
        sample_timestamp
    )

    print(
        f"Lender {lender}: "
        f"{len(prior_events)} earlier events"
    )

    # Verify that every retrieved event is strictly earlier
    if len(prior_events) > 0:
        prior_timestamps = all_graph_events.loc[
            prior_events, "timestamp"
        ].to_numpy()

        assert np.all(
            prior_timestamps < sample_timestamp
        )

        assert sample_event_id not in prior_events

print("\nHistorical neighbour retrieval successful!")


### Original notebook cell 104


In [ ]:

# Step 53: Implement time-restricted historical-neighbour sampling

import numpy as np

MAX_HISTORICAL_NEIGHBOURS = 32
SAMPLING_SEED = 42

def sample_prior_lender_events(
    lender_index,
    target_timestamp,
    max_neighbours=MAX_HISTORICAL_NEIGHBOURS,
    seed=SAMPLING_SEED
):
    """
    Sample earlier borrowing events connected to a lender.

    Only events strictly before target_timestamp are eligible.
    Sampling is deterministic for the same inputs.
    """
    prior_events = get_prior_lender_events(
        lender_index,
        target_timestamp
    )

    if len(prior_events) <= max_neighbours:
        return prior_events.copy()

    # Reproducible random sampling without replacement
    seed_sequence = np.random.SeedSequence([
        seed,
        int(lender_index),
        int(target_timestamp)
    ])

    rng = np.random.default_rng(seed_sequence)

    sampled = rng.choice(
        prior_events,
        size=max_neighbours,
        replace=False
    )

    return np.sort(sampled)


# Test sampling using the validation event from Step 51
print("HISTORICAL NEIGHBOUR SAMPLING")
print("-----------------------------")
print("Target event ID:", sample_event_id)
print("Maximum historical neighbours:", MAX_HISTORICAL_NEIGHBOURS)

for lender in selected_lenders:
    sampled_events = sample_prior_lender_events(
        int(lender),
        sample_timestamp
    )

    # Repeat to verify reproducibility
    repeated_sample = sample_prior_lender_events(
        int(lender),
        sample_timestamp
    )

    assert np.array_equal(
        sampled_events,
        repeated_sample
    )

    assert len(sampled_events) <= MAX_HISTORICAL_NEIGHBOURS
    assert len(np.unique(sampled_events)) == len(sampled_events)

    if len(sampled_events) > 0:
        sampled_timestamps = all_graph_events.loc[
            sampled_events, "timestamp"
        ].to_numpy()

        assert np.all(sampled_timestamps < sample_timestamp)
        assert sample_event_id not in sampled_events

    print(
        f"Lender {lender}: "
        f"{len(sampled_events)} sampled historical events"
    )

print("\nHistorical neighbour sampling successful!")


### Original notebook cell 106


In [ ]:

# Step 54: Build a sampled historical neighbourhood for one event

import numpy as np

def build_event_neighbourhood(
    target_event_id,
    max_neighbours=MAX_HISTORICAL_NEIGHBOURS
):
    """
    Construct a two-hop historical neighbourhood:
    target event -> participating lenders -> earlier events.
    """
    target_row = all_graph_events.iloc[int(target_event_id)]
    target_timestamp = int(target_row["timestamp"])

    # Retrieve lenders funding the target event
    target_lenders = temporal_adjacency.loc[
        temporal_adjacency["event_node_index"] == target_event_id,
        "lender_node_index"
    ].to_numpy(dtype=np.int64)

    # Store lender-to-event relationships
    historical_edges = []

    for lender in target_lenders:
        earlier_events = sample_prior_lender_events(
            int(lender),
            target_timestamp,
            max_neighbours=max_neighbours
        )

        for earlier_event in earlier_events:
            historical_edges.append(
                (int(lender), int(earlier_event))
            )

    historical_edges = np.asarray(
        historical_edges,
        dtype=np.int64
    ).reshape(-1, 2)

    return {
        "target_event_id": int(target_event_id),
        "target_timestamp": target_timestamp,
        "target_lenders": np.unique(target_lenders),
        "historical_edges": historical_edges,
        "historical_event_ids": np.unique(
            historical_edges[:, 1]
        )
    }


sample_neighbourhood = build_event_neighbourhood(
    sample_event_id
)

print("SAMPLED HISTORICAL EVENT NEIGHBOURHOOD")
print("--------------------------------------")
print("Target event:", sample_neighbourhood["target_event_id"])
print("Participating lenders:",
      len(sample_neighbourhood["target_lenders"]))
print("Sampled historical edges:",
      len(sample_neighbourhood["historical_edges"]))
print("Unique historical events:",
      len(sample_neighbourhood["historical_event_ids"]))

# Verify that every historical event is earlier than the target
historical_ids = sample_neighbourhood["historical_event_ids"]

if len(historical_ids) > 0:
    historical_times = all_graph_events.iloc[
        historical_ids
    ]["timestamp"].to_numpy()

    assert np.all(
        historical_times < sample_neighbourhood["target_timestamp"]
    )

    assert sample_event_id not in historical_ids

assert len(sample_neighbourhood["target_lenders"]) == len(sample_lenders)

assert len(sample_neighbourhood["historical_edges"]) <= (
    len(sample_lenders) * MAX_HISTORICAL_NEIGHBOURS
)

print("\nHistorical neighbourhood construction successful!")


### Original notebook cell 108


In [ ]:

# Step 55: Convert a sampled neighbourhood into HeteroData

import numpy as np
import torch
from torch_geometric.data import HeteroData

def neighbourhood_to_heterodata(neighbourhood):
    target_id = neighbourhood["target_event_id"]
    target_lenders = neighbourhood["target_lenders"]
    historical_edges = neighbourhood["historical_edges"]
    historical_ids = neighbourhood["historical_event_ids"]

    # Local event nodes: target first, then unique historical events
    local_event_ids = np.concatenate([
        np.array([target_id], dtype=np.int64),
        historical_ids.astype(np.int64)
    ])

    local_lender_ids = np.unique(target_lenders)

    # Map global node IDs to positions in this small graph
    event_lookup = {
        int(node_id): i
        for i, node_id in enumerate(local_event_ids)
    }

    lender_lookup = {
        int(node_id): i
        for i, node_id in enumerate(local_lender_ids)
    }

    # Include current target funding relationships
    current_edges = [
        (int(lender), int(target_id))
        for lender in target_lenders
    ]

    # Include sampled historical relationships
    all_edges = current_edges + [
        (int(lender), int(event))
        for lender, event in historical_edges
    ]

    local_edges = torch.tensor(
        [
            [
                lender_lookup[lender],
                event_lookup[event]
            ]
            for lender, event in all_edges
        ],
        dtype=torch.long
    ).T.contiguous()

    graph = HeteroData()

    graph["lender"].x = lender_x[
        torch.tensor(local_lender_ids, dtype=torch.long)
    ]

    graph["event"].x = event_x[
        torch.tensor(local_event_ids, dtype=torch.long)
    ]

    graph["lender", "funds", "event"].edge_index = local_edges

    graph["event", "funded_by", "lender"].edge_index = (
        local_edges.flip(0).contiguous()
    )

    # The first local event node is the target
    graph["event"].target_mask = torch.zeros(
        len(local_event_ids),
        dtype=torch.bool
    )
    graph["event"].target_mask[0] = True

    return graph


sample_hetero_graph = neighbourhood_to_heterodata(
    sample_neighbourhood
)

print("SAMPLED HETEROGENEOUS GRAPH")
print("---------------------------")
print(sample_hetero_graph)

print("\nNode counts:")
print("Lenders:", sample_hetero_graph["lender"].num_nodes)
print("Events:", sample_hetero_graph["event"].num_nodes)

print("\nDirected edge counts:")
print(
    "Lender to event:",
    sample_hetero_graph["lender", "funds", "event"].num_edges
)
print(
    "Event to lender:",
    sample_hetero_graph["event", "funded_by", "lender"].num_edges
)

print("\nTarget event nodes:",
      sample_hetero_graph["event"].target_mask.sum().item())

assert sample_hetero_graph.validate(raise_on_error=True)
assert sample_hetero_graph["event"].target_mask.sum() == 1
assert sample_hetero_graph["event"].num_nodes == 238
assert sample_hetero_graph["lender"].num_nodes == 10

print("\nHeterogeneous graph construction successful!")


### Original notebook cell 110


In [ ]:

# Step 56: Test heterogeneous GraphSAGE on Prosper data

import torch
import torch.nn.functional as F

from torch import nn
from torch_geometric.nn import HeteroConv, SAGEConv

torch.manual_seed(42)

class ProsperGraphSAGE(nn.Module):
    def __init__(self, hidden_channels=32):
        super().__init__()

        # Project different node feature dimensions
        # into a common hidden representation
        self.lender_encoder = nn.Linear(1, hidden_channels)
        self.event_encoder = nn.Linear(11, hidden_channels)

        # First message-passing layer
        self.conv1 = HeteroConv({
            ("lender", "funds", "event"):
                SAGEConv((-1, -1), hidden_channels),
            ("event", "funded_by", "lender"):
                SAGEConv((-1, -1), hidden_channels)
        }, aggr="sum")

        # Second message-passing layer
        self.conv2 = HeteroConv({
            ("lender", "funds", "event"):
                SAGEConv((-1, -1), hidden_channels),
            ("event", "funded_by", "lender"):
                SAGEConv((-1, -1), hidden_channels)
        }, aggr="sum")

        # Binary classification output
        self.classifier = nn.Linear(hidden_channels, 1)

    def forward(self, data):
        x_dict = {
            "lender": F.relu(
                self.lender_encoder(data["lender"].x)
            ),
            "event": F.relu(
                self.event_encoder(data["event"].x)
            )
        }

        x_dict = self.conv1(
            x_dict,
            data.edge_index_dict
        )

        x_dict = {
            key: F.relu(value)
            for key, value in x_dict.items()
        }

        x_dict = self.conv2(
            x_dict,
            data.edge_index_dict
        )

        # Only classify the target borrowing event
        target_embeddings = x_dict["event"][
            data["event"].target_mask
        ]

        logits = self.classifier(
            target_embeddings
        ).squeeze(-1)

        return logits


# Initialise the model
gnn_model = ProsperGraphSAGE(hidden_channels=32)

# Test without training
gnn_model.eval()

with torch.no_grad():
    sample_logits = gnn_model(sample_hetero_graph)
    sample_probabilities = torch.sigmoid(sample_logits)

print("PROSPER GRAPHSAGE FORWARD PASS")
print("------------------------------")
print("Output shape:", tuple(sample_logits.shape))
print("Raw prediction (logit):", sample_logits.item())
print("Predicted default probability:",
      sample_probabilities.item())

assert sample_logits.shape == (1,)
assert torch.isfinite(sample_logits).all()
assert 0 <= sample_probabilities.item() <= 1

print("\nGraphSAGE forward pass successful!")


### Original notebook cell 116


In [ ]:

# Step 59: Prepare chronological event IDs for GraphSAGE

import numpy as np
import torch

# Retrieve event IDs using the existing chronological masks
train_event_ids = torch.where(
    train_node_mask
)[0].cpu().numpy().astype(np.int64)

validation_event_ids = torch.where(
    validation_node_mask
)[0].cpu().numpy().astype(np.int64)

test_event_ids = torch.where(
    test_node_mask
)[0].cpu().numpy().astype(np.int64)

# Check that the expected events were selected
assert len(train_event_ids) == 17361
assert len(validation_event_ids) == 4379
assert len(test_event_ids) == 5545

# Confirm that partitions do not overlap
assert len(
    np.intersect1d(train_event_ids, validation_event_ids)
) == 0

assert len(
    np.intersect1d(train_event_ids, test_event_ids)
) == 0

assert len(
    np.intersect1d(validation_event_ids, test_event_ids)
) == 0

# Confirm all selected events have valid labels
for partition_ids in [
    train_event_ids,
    validation_event_ids,
    test_event_ids
]:
    labels = event_y[
        torch.from_numpy(partition_ids)
    ]

    assert torch.all(
        (labels == 0) | (labels == 1)
    )

# Verify chronological boundaries
train_times = all_graph_events.iloc[
    train_event_ids
]["timestamp"].to_numpy()

validation_times = all_graph_events.iloc[
    validation_event_ids
]["timestamp"].to_numpy()

test_times = all_graph_events.iloc[
    test_event_ids
]["timestamp"].to_numpy()

assert train_times.max() < validation_times.min()
assert validation_times.max() < test_times.min()

print("GRAPHSAGE CHRONOLOGICAL TRAINING PARTITIONS")
print("------------------------------------------")
print("Training events:", len(train_event_ids))
print("Validation events:", len(validation_event_ids))
print("Test events:", len(test_event_ids))

print("\nChronological separation:")
print("Train before validation:",
      train_times.max() < validation_times.min())
print("Validation before test:",
      validation_times.max() < test_times.min())

print("\nPartition overlap: None")
print("All selected events have valid binary labels.")

print("\nGraphSAGE training partition preparation successful!")


### Original notebook cell 118


In [ ]:

# Step 60: Build an efficient event-to-lender lookup

import numpy as np

# Use the existing distinct lender-event relationships
event_lender_pairs = indexed_graph_edges[
    ["event_node_index", "lender_node_index"]
].to_numpy(dtype=np.int64)

# Sort relationships by event ID, then lender ID
sort_order = np.lexsort((
    event_lender_pairs[:, 1],
    event_lender_pairs[:, 0]
))

sorted_event_lender_pairs = event_lender_pairs[sort_order]

event_lookup_ids = sorted_event_lender_pairs[:, 0]
event_lookup_lenders = sorted_event_lender_pairs[:, 1]

# Count participating lenders for each borrowing event
event_lender_counts = np.bincount(
    event_lookup_ids,
    minlength=len(all_graph_events)
)

# Build offsets to locate each event's lenders
event_lender_offsets = np.zeros(
    len(all_graph_events) + 1,
    dtype=np.int64
)

event_lender_offsets[1:] = np.cumsum(
    event_lender_counts
)

def get_event_lenders(event_id):
    """
    Return the participating lender indices for an event
    using the precomputed lookup.
    """
    start = event_lender_offsets[int(event_id)]
    end = event_lender_offsets[int(event_id) + 1]

    return event_lookup_lenders[start:end]


# Validate against the previously constructed neighbourhood
retrieved_lenders = get_event_lenders(sample_event_id)

assert np.array_equal(
    retrieved_lenders,
    np.sort(sample_neighbourhood["target_lenders"])
)

assert event_lender_offsets[-1] == len(indexed_graph_edges)

assert len(event_lender_counts) == len(all_graph_events)

# Verify that all supervised events have participating lenders
supervised_ids = np.concatenate([
    train_event_ids,
    validation_event_ids,
    test_event_ids
])

assert np.all(event_lender_counts[supervised_ids] > 0)

print("FAST EVENT-TO-LENDER LOOKUP")
print("---------------------------")
print("Total borrowing events:", len(all_graph_events))
print("Indexed funding relationships:", len(event_lookup_lenders))
print("Lookup offset entries:", len(event_lender_offsets))

print("\nSample event:", sample_event_id)
print("Participating lenders:", len(retrieved_lenders))
print("Matches previous neighbourhood:",
      np.array_equal(
          retrieved_lenders,
          np.sort(sample_neighbourhood["target_lenders"])
      ))

print("\nLenders per supervised event:")
print(
    np.percentile(
        event_lender_counts[supervised_ids],
        [0, 25, 50, 75, 100]
    )
)

print("\nFast event-to-lender lookup successful!")


### Original notebook cell 120


In [ ]:

# Step 61: Bounded temporal neighbourhood sampling

import numpy as np

MAX_TARGET_LENDERS = 16
MAX_HISTORICAL_NEIGHBOURS = 32
SAMPLING_SEED = 42

def build_bounded_event_neighbourhood(
    target_event_id,
    max_lenders=MAX_TARGET_LENDERS,
    max_historical=MAX_HISTORICAL_NEIGHBOURS,
    seed=SAMPLING_SEED
):
    """
    Build a bounded historical neighbourhood for one target event.

    All historical events must occur strictly before the
    target event's timestamp.
    """

    target_event_id = int(target_event_id)

    target_timestamp = int(
        all_graph_events.iloc[target_event_id]["timestamp"]
    )

    # Fast lookup of current participating lenders
    participating_lenders = get_event_lenders(
        target_event_id
    )

    # Reproducibly sample participating lenders if necessary
    if len(participating_lenders) > max_lenders:
        rng = np.random.default_rng(
            np.random.SeedSequence([
                seed,
                target_event_id
            ])
        )

        selected_lenders = np.sort(
            rng.choice(
                participating_lenders,
                size=max_lenders,
                replace=False
            )
        )
    else:
        selected_lenders = participating_lenders.copy()

    # Retrieve earlier events for each selected lender
    historical_edges = []

    for lender in selected_lenders:
        prior_events = sample_prior_lender_events(
            lender_index=int(lender),
            target_timestamp=target_timestamp,
            max_neighbours=max_historical,
            seed=seed
        )

        for prior_event in prior_events:
            historical_edges.append(
                (int(lender), int(prior_event))
            )

    historical_edges = np.asarray(
        historical_edges,
        dtype=np.int64
    ).reshape(-1, 2)

    historical_event_ids = np.unique(
        historical_edges[:, 1]
    )

    return {
        "target_event_id": target_event_id,
        "target_timestamp": target_timestamp,
        "target_lenders": selected_lenders,
        "historical_edges": historical_edges,
        "historical_event_ids": historical_event_ids
    }


# Test on an event with many participating lenders
largest_event_id = int(
    supervised_ids[
        np.argmax(event_lender_counts[supervised_ids])
    ]
)

large_neighbourhood = build_bounded_event_neighbourhood(
    largest_event_id
)

# Rebuild to verify reproducibility
repeated_neighbourhood = build_bounded_event_neighbourhood(
    largest_event_id
)

assert np.array_equal(
    large_neighbourhood["target_lenders"],
    repeated_neighbourhood["target_lenders"]
)

assert np.array_equal(
    large_neighbourhood["historical_edges"],
    repeated_neighbourhood["historical_edges"]
)

assert len(large_neighbourhood["target_lenders"]) <= 16

assert len(large_neighbourhood["historical_edges"]) <= 512

# Check chronological restrictions
historical_ids = large_neighbourhood["historical_event_ids"]

if len(historical_ids) > 0:
    historical_timestamps = all_graph_events.iloc[
        historical_ids
    ]["timestamp"].to_numpy()

    assert np.all(
        historical_timestamps <
        large_neighbourhood["target_timestamp"]
    )

    assert largest_event_id not in historical_ids

print("BOUNDED TEMPORAL NEIGHBOURHOOD SAMPLING")
print("---------------------------------------")
print("Target event ID:", largest_event_id)
print("Original participating lenders:",
      event_lender_counts[largest_event_id])
print("Sampled participating lenders:",
      len(large_neighbourhood["target_lenders"]))
print("Sampled historical edges:",
      len(large_neighbourhood["historical_edges"]))
print("Unique historical events:",
      len(large_neighbourhood["historical_event_ids"]))

print("\nMaximum target lenders:", MAX_TARGET_LENDERS)
print("Maximum history per lender:", MAX_HISTORICAL_NEIGHBOURS)
print("Maximum historical edges:",
      MAX_TARGET_LENDERS * MAX_HISTORICAL_NEIGHBOURS)

print("\nReproducibility and chronological checks passed!")


### Original notebook cell 122


In [ ]:

# Step 62: Construct a temporally isolated GraphSAGE mini-batch

import numpy as np
import torch
from torch_geometric.data import Batch

def build_graphsage_batch(event_ids):
    """
    Construct a batch of independent, time-restricted
    heterogeneous neighbourhoods.
    """

    graphs = []

    for event_id in event_ids:
        neighbourhood = build_bounded_event_neighbourhood(
            int(event_id)
        )

        graph = neighbourhood_to_heterodata(
            neighbourhood
        )

        # Store the target event's supervised label
        # separately from the input node features
        graph["event"].target_y = event_y[
            int(event_id)
        ].reshape(1).float()

        graphs.append(graph)

    # PyG offsets local indices and keeps graphs disconnected
    batch = Batch.from_data_list(graphs)

    return batch


# Select a small training-only batch
sample_train_ids = train_event_ids[:4]

sample_train_batch = build_graphsage_batch(
    sample_train_ids
)

print("GRAPHSAGE MINI-BATCH CONSTRUCTION")
print("---------------------------------")
print("Training event IDs:", sample_train_ids.tolist())
print("Number of graphs:", sample_train_batch.num_graphs)

print("\nBatched graph:")
print(sample_train_batch)

print("\nTotal lender nodes:",
      sample_train_batch["lender"].num_nodes)

print("Total event nodes:",
      sample_train_batch["event"].num_nodes)

print("Total target nodes:",
      sample_train_batch["event"].target_mask.sum().item())

print("Target labels:",
      sample_train_batch["event"].target_y.tolist())

# Validation checks
assert sample_train_batch.num_graphs == 4

assert (
    sample_train_batch["event"].target_mask.sum().item()
    == 4
)

assert sample_train_batch["event"].target_y.numel() == 4

assert sample_train_batch.validate(raise_on_error=True)

# Verify graph isolation: every edge must connect nodes
# belonging to the same sampled neighbourhood
for edge_type in sample_train_batch.edge_types:
    source_type, _, destination_type = edge_type

    edge_index = sample_train_batch[edge_type].edge_index

    source_graph_ids = sample_train_batch[
        source_type
    ].batch[edge_index[0]]

    destination_graph_ids = sample_train_batch[
        destination_type
    ].batch[edge_index[1]]

    assert torch.equal(
        source_graph_ids,
        destination_graph_ids
    ), f"Cross-graph connection detected: {edge_type}"

# Verify that all four events belong to training
assert np.all(
    np.isin(sample_train_ids, train_event_ids)
)

print("\nAll neighbourhoods are isolated.")
print("All target labels belong to training.")
print("\nGraphSAGE mini-batch construction successful!")
